# Lab 0 Task 0.2.1 — Transfer Learning from ImageNet
In this section, we use AlexNet on CIFAR-10 in two settings:
1. Fine-tuning the whole network
2. Feature extraction with pretrained weights

## Setup & Imports

In [1]:
import warnings
import copy
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.models import alexnet, AlexNet_Weights
from typing import List, Dict, Tuple, Any

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make results reproducible
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

Using device: cuda


## Load CIFAR-10

We apply ImageNet normalization and resize all images to 224x224 so they match AlexNet's expected input.

In [2]:
IMG_SIZE = 224
BATCH_SIZE = 256

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

transform_alexnet = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

train_dataset = torchvision.datasets.CIFAR10(
    root="../data", train=True, download=True, transform=transform_alexnet
)
test_dataset = torchvision.datasets.CIFAR10(
    root="../data", train=False, download=True, transform=transform_alexnet
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

classes: list[str] = train_dataset.classes
print("Classes:", classes)

/usr/local/lib/python3.11/dist-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## AlexNet model
We replace the last classifier layer so the output dimension becomes 10.

In [3]:
def build_alexnet(pretrained=False, freeze_features=False):
    weights = AlexNet_Weights.DEFAULT if pretrained else None
    model = alexnet(weights=weights)

    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(in_features, 10)

    if freeze_features:
        for p in model.features.parameters():
            p.requires_grad = False
        for layer in model.classifier[:-1]:
            for p in layer.parameters():
                p.requires_grad = False

    return model

In [4]:
def train(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> Tuple[float, float]:
    """Run one full pass over `loader` in training mode."""
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct      += (outputs.argmax(dim=1) == labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def validate(
    model: nn.Module,
    loader: DataLoader,
) -> Tuple[float, float]:
    """Run one full pass over `loader` in evaluation mode."""
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            correct      += (outputs.argmax(dim=1) == labels).sum().item()
            total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def fit(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    wandb_kwargs: Dict[str, Any],
    num_epochs: int = 10,
) -> Dict[str, List[float]]:
    """Train `model` for `num_epochs`, validating after every epoch.

    Initialises and closes a wandb run for the duration of training.
    Saves the weights that achieved the best validation accuracy and
    restores them at the end.

    Returns
    -------
    history : dict with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """

    with wandb.init(**wandb_kwargs):

        best_val_acc = 0.0
        best_state   = None
        history: dict[str, list[float]] = {
            "train_loss": [], "val_loss": [],
            "train_acc":  [], "val_acc":  [],
        }
    
        for epoch in range(1, num_epochs + 1):
            train_loss, train_acc = train(model, train_loader, optimizer)
            val_loss,   val_acc   = validate(model, val_loader)
    
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)
    
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
            print(
                f"Epoch {epoch:>{len(str(num_epochs))}}/{num_epochs} | "
                f"train loss {train_loss:.4f}, train acc {train_acc:.2f}% | "
                f"val loss {val_loss:.4f}, val acc {val_acc:.2f}%"
            )
    
            wandb.log({
                "Training Loss": train_loss,
                "Training Accuracy": train_acc,
                "Validation Loss": val_loss,
                "Validation Accuracy": val_acc
            })
    
    # Restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best weights (val acc {best_val_acc:.2f}%)")

    return history

## Experiment A — Fine-tuning AlexNet on CIFAR-10
Here we train the whole network end-to-end.

In [5]:
criterion = nn.CrossEntropyLoss()

NUM_EPOCHS = 5
LEARNING_RATE = 1e-4

model_ft = build_alexnet(pretrained=False, freeze_features=False).to(device)
optimizer_ft = optim.Adam(model_ft.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="AlexNet Fine-tuning",
    tags=["Task 0.2.1", "Fine-tuning"],
    config={"pretrained": False, "freeze_features": False, "optimizer": "Adam",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_ft = fit(model_ft, optimizer_ft, train_loader, test_loader,
                 wandb_kwargs=wandb_kwargs, num_epochs=NUM_EPOCHS)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: oscar-engelmark (d7047e-group12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/5 | train loss 1.6709, train acc 38.64% | val loss 1.3863, val acc 48.50%
Epoch 2/5 | train loss 1.2320, train acc 55.47% | val loss 1.0515, val acc 62.83%
Epoch 3/5 | train loss 1.0073, train acc 64.05% | val loss 0.9248, val acc 68.13%
Epoch 4/5 | train loss 0.8514, train acc 70.06% | val loss 0.8641, val acc 70.02%
Epoch 5/5 | train loss 0.7520, train acc 73.55% | val loss 0.7501, val acc 74.47%


Training Accuracy,▁▄▆▇█
Training Loss,█▅▃▂▁
Validation Accuracy,▁▅▆▇█
Validation Loss,█▄▃▂▁
Training Accuracy,73.548
Training Loss,0.75204
Validation Accuracy,74.47
Validation Loss,0.75014



Restored best weights (val acc 74.47%)


## Experiment B — Feature extraction with pretrained AlexNet
Here we freeze the backbone and train only the final classifier layer.

In [6]:
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3

model_fe = build_alexnet(pretrained=True, freeze_features=True).to(device)
optimizer_fe = optim.Adam([p for p in model_fe.parameters() if p.requires_grad], lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="AlexNet Feature Extraction",
    tags=["Task 0.2.1", "Feature Extraction"],
    config={"pretrained": True, "freeze_features": True, "optimizer": "Adam",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_fe = fit(model_fe, optimizer_fe, train_loader, test_loader,
                 wandb_kwargs=wandb_kwargs, num_epochs=NUM_EPOCHS)

Epoch 1/5 | train loss 0.7395, train acc 74.24% | val loss 0.5522, val acc 80.50%
Epoch 2/5 | train loss 0.5962, train acc 78.89% | val loss 0.5047, val acc 82.47%
Epoch 3/5 | train loss 0.5615, train acc 80.26% | val loss 0.5045, val acc 82.33%
Epoch 4/5 | train loss 0.5513, train acc 80.45% | val loss 0.4888, val acc 82.73%
Epoch 5/5 | train loss 0.5376, train acc 81.05% | val loss 0.4844, val acc 83.07%


Training Accuracy,▁▆▇▇█
Training Loss,█▃▂▁▁
Validation Accuracy,▁▆▆▇█
Validation Loss,█▃▃▁▁
Training Accuracy,81.05
Training Loss,0.5376
Validation Accuracy,83.07
Validation Loss,0.48441



Restored best weights (val acc 83.07%)


## Brief comparison
Fine-tuning updates the whole AlexNet, so the model can adapt to CIFAR-10 more strongly.
Feature extraction keeps pretrained layers frozen, so only the new classifier learns.
That is why fine-tuning usually performs better when the target dataset is different from ImageNet.

In [7]:
print("===== Final Comparison (Accuracy) =====")
print(f"Fine-tuning     best val acc: {max(history_ft['val_acc']):.2f}%")
print(f"Feature extract best val acc: {max(history_fe['val_acc']):.2f}%")

===== Final Comparison (Accuracy) =====


NameError: name 'best_acc_ft' is not defined